# 03 · Anotación de tipos celulares
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kur1sutaru/ISMB-LATAM/blob/main/notebooks/es/03_cell_type_annotation.ipynb)

**De Células a Clústeres: Introducción al Análisis de Single-Cell RNA-seq en Python**  
ISCB-LATAM SolBio SPBBC 2026 · Lima, Perú · 9 de noviembre de 2026  
Instructora: Dra. Cristal Villalba Silva (UT Health San Antonio) · [github.com/Kur1sutaru](https://github.com/Kur1sutaru)

⏱ **Tiempo en el tutorial:** 11:00–11:35 (35 min)

**Al final de este notebook podrás:**
- Encontrar genes marcadores de cada clúster y anotarlos manualmente con marcadores conocidos
- Usar una base de datos de marcadores (**PanglaoDB**) para puntuar tipos celulares
- Anotar automáticamente con un clasificador preentrenado (**CellTypist**) y con una referencia (`sc.tl.ingest`)
- Saber dónde encontrar referencias y registrar anotaciones: **CZ CELLxGENE**, **Single Cell Portal**, **Cell Annotation Platform (CAP)**, Human Cell Atlas, Cell Ontology
- Comparar métodos de anotación y decidir las etiquetas finales

> 🟢 Las secciones **Core** se ven en vivo. 🔎 Las secciones **Extra** son para estudiar después del tutorial.

## 0. Configuración 🟢
Ejecuta primero la celda de abajo (▶ o `Shift + Enter`). En Google Colab instala los paquetes (~1 min);
en tu computadora asume que ya ejecutaste `pip install -r requirements.txt`.

**Opcional:** cambia `USE_DRIVE = True` para guardar tus resultados en Google Drive y reutilizarlos en
los siguientes notebooks. Si lo dejas en `False`, cada notebook reconstruye lo que necesita automáticamente.

In [ ]:
import os, sys, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "scanpy[leiden,scrublet]>=1.10" "decoupler>=2.1,<3" "celltypist>=1.7"

USE_DRIVE = False  # True = guardar resultados en Google Drive
WORKDIR = "sc_tutorial"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/sc_tutorial"
os.makedirs(WORKDIR, exist_ok=True)

# Funciones auxiliares del tutorial (descarga + checkpoints)
if os.path.exists("../../scripts/sc_tutorial.py"):          # ejecutando desde un clon local
    sys.path.insert(0, os.path.abspath("../../scripts"))
elif not os.path.exists("sc_tutorial.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Kur1sutaru/ISMB-LATAM/main/scripts/sc_tutorial.py", "sc_tutorial.py")
import sc_tutorial as tut

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import decoupler as dc
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sc.settings.figdir = os.path.join(WORKDIR, "figures")
print("scanpy", sc.__version__, "| working directory:", os.path.abspath(WORKDIR))

In [ ]:
adata = tut.get_checkpoint("clustered", WORKDIR)
sc.pl.umap(adata, color="leiden", legend_loc="on data", title="Leiden clusters")

## 1. Tres formas de anotar 🟢

| Estrategia | Cómo | Ventajas | Desventajas |
|---|---|---|---|
| **Manual (marcadores)** | Buscar genes enriquecidos en cada clúster y compararlos con marcadores conocidos | Transparente, usa tu experiencia | Lenta, subjetiva, necesita buenos marcadores |
| **Bases de datos de marcadores** | Puntuar cada célula con listas curadas (PanglaoDB, CellMarker) | Rápida, no necesita datos de referencia | Listas genéricas, el contexto del tejido importa |
| **Por referencia / clasificadores** | Transferir etiquetas desde un atlas anotado (CellTypist, ingest, scANVI, Azimuth, SingleR) | Rápida, reproducible, detallada | Tan buena como la referencia; no encuentra tipos nuevos |

En la práctica se **combinan** y la concordancia entre métodos se usa como evidencia.

## 2. Genes marcadores a partir de los datos 🟢
`sc.tl.rank_genes_groups` compara cada clúster contra todas las demás células (prueba de Wilcoxon sobre los
valores log-normalizados guardados en `adata.raw`).

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=4, standard_scale="var")

In [ ]:
markers_df = sc.get.rank_genes_groups_df(adata, group=None)
top = (markers_df.query("logfoldchanges > 1 and pvals_adj < 0.05")
       .groupby("group", observed=True).head(5)
       .groupby("group", observed=True)["names"].apply(", ".join))
top.to_frame("top 5 markers")

## 3. Anotación manual con marcadores conocidos 🟢
Marcadores clásicos de PBMC (de los tutoriales de Seurat/Scanpy y la literatura):

In [ ]:
pbmc_markers = tut.PBMC_MARKERS
pd.Series({k: ", ".join(v) for k, v in pbmc_markers.items()}).to_frame("markers")

In [ ]:
sc.pl.dotplot(adata, pbmc_markers, groupby="leiden", standard_scale="var")

**Cómo leer un dot plot:** el **tamaño** del punto = % de células del clúster que expresan el gen; el **color** =
expresión media (escalada por gen). Un buen marcador es grande y oscuro en un grupo y pequeño en los demás.

Para que sea reproducible, **puntuamos** cada conjunto de marcadores por célula (`sc.tl.score_genes`) y asignamos a
cada clúster la etiqueta con el puntaje promedio más alto.

In [ ]:
adata = tut.annotate_by_markers(adata, pbmc_markers, cluster_key="leiden", key_added="cell_type")
score_cols = [c for c in adata.obs.columns if c.startswith("score_")]
display(adata.obs.groupby("leiden", observed=True)[score_cols].mean().round(2))
pd.crosstab(adata.obs["leiden"], adata.obs["cell_type"])

Si no estás de acuerdo con una etiqueta (¡esa es tu experiencia!), corrígela con un diccionario:
```python
manual = {"0": "CD4 T", "1": "CD14 Mono", ...}      # clúster -> tipo celular
adata.obs["cell_type"] = adata.obs["leiden"].map(manual).astype("category")
```

In [ ]:
sc.pl.umap(adata, color=["leiden", "cell_type"], legend_loc="on data", ncols=2)

## 4. Base de datos de marcadores: PanglaoDB 🟢
[**PanglaoDB**](https://panglaodb.se) reúne >8.000 asociaciones marcador–tipo celular de >1.000 conjuntos de
single-cell (humano y ratón). La descargamos a través de **OmniPath** con `decoupler` y puntuamos cada célula con un
**modelo lineal univariado (ULM)**: para cada tipo celular, ¿qué tanto explican sus marcadores la expresión de la célula?

In [ ]:
panglao = tut.cached_table("panglaodb_human", lambda: dc.op.resource("PanglaoDB", organism="human"), WORKDIR)
is_true = lambda s: s.astype(str).str.lower().isin(["true", "1", "1.0"])
markers_db = panglao[is_true(panglao["human"]) & is_true(panglao["canonical_marker"])
                     & (pd.to_numeric(panglao["human_sensitivity"], errors="coerce") > 0.5)]
markers_db = (markers_db.rename(columns={"cell_type": "source", "genesymbol": "target"})
              [["source", "target"]].drop_duplicates())
print(f"{markers_db['source'].nunique()} cell types, {len(markers_db)} markers")
markers_db.head()

In [ ]:
dc.mt.ulm(adata, net=markers_db, tmin=3, raw=True)
panglao_scores = dc.pp.get_obsm(adata, "score_ulm")
ranked = dc.tl.rankby_group(panglao_scores, groupby="leiden")
top_panglao = (ranked[ranked["stat"] > 0].sort_values("stat", ascending=False)
               .groupby("group", observed=True).head(3)
               .groupby("group", observed=True)["name"].apply(" | ".join))
top_panglao.to_frame("top 3 PanglaoDB cell types")

In [ ]:
best = ranked.sort_values("stat", ascending=False).groupby("group", observed=True).head(1).set_index("group")["name"]
adata.obs["panglaodb"] = adata.obs["leiden"].map(best).astype("category")
show = [c for c in ["T cells", "NK cells", "B cells", "Monocytes", "Dendritic cells", "Platelets"]
        if c in panglao_scores.var_names]
sc.pl.umap(panglao_scores, color=show, cmap="RdBu_r", vcenter=0, ncols=3)

## 5. Clasificador automático: CellTypist 🟢
[**CellTypist**](https://www.celltypist.org) es un clasificador de regresión logística entrenado con grandes atlas
inmunes (Domínguez Conde *et al.*, *Science* 2022). `Immune_All_Low` predice ~98 tipos inmunes detallados;
`Immune_All_High` da ~32 más generales. Existen muchos otros modelos (pulmón, intestino, cerebro, tejidos en desarrollo, ratón...).

CellTypist espera **log1p(conteos por 10k)**, que es justo lo que guardamos en `adata.raw`.
`majority_voting=True` suaviza las predicciones por célula dentro de cada clúster.

In [ ]:
import celltypist
from celltypist import models

model_name = "Immune_All_Low.pkl"
if not os.path.exists(models.get_model_path(model_name)):
    models.download_models(model=[model_name])

adata_ct = adata.raw.to_adata()     # todos los genes, log1p(CP10k)
pred = celltypist.annotate(adata_ct, model=model_name, majority_voting=True,
                           over_clustering="leiden")
adata.obs["celltypist"] = pred.predicted_labels["majority_voting"].astype("category").values
adata.obs["celltypist_conf"] = pred.probability_matrix.max(axis=1).values
sc.pl.umap(adata, color=["celltypist", "celltypist_conf"], ncols=2, wspace=0.6)

## 6. Mapeo a una referencia con `sc.tl.ingest` 🟢
`ingest` proyecta tus células sobre el PCA/UMAP de una **referencia anotada** y transfiere las etiquetas de las células
de referencia más cercanas. Aquí la referencia es `pbmc68k_reduced`, un pequeño subconjunto anotado de 68k PBMCs que
viene con Scanpy. La misma idea escala a grandes atlas con **scANVI**, **scArches** o **Azimuth** (R).

In [ ]:
ref = sc.datasets.pbmc68k_reduced()
shared = ref.var_names.intersection(adata.var_names)
ref = ref[:, shared].copy()
query = adata[:, shared].copy()
n_pcs = min(30, len(shared) - 1)
sc.pp.pca(ref, n_comps=n_pcs)
sc.pp.neighbors(ref, n_pcs=n_pcs)
sc.tl.umap(ref)
sc.tl.ingest(query, ref, obs="bulk_labels")
adata.obs["ingest"] = query.obs["bulk_labels"].values
print(len(shared), "shared genes")
sc.pl.umap(adata, color="ingest")

## 7. Dónde encontrar referencias y registrar anotaciones 🟢

| Recurso | Qué ofrece | Cómo usarlo desde Python |
|---|---|---|
| [**PanglaoDB**](https://panglaodb.se) | Genes marcadores curados (humano, ratón) | `decoupler.op.resource("PanglaoDB")` (arriba) |
| [**CellMarker 2.0**](http://bio-bigdata.hrbmu.edu.cn/CellMarker/) | Marcadores curados manualmente de >100k publicaciones | Descargar la tabla → mismo enfoque ULM |
| [**CZ CELLxGENE Discover**](https://cellxgene.cziscience.com) | >1.000 conjuntos estandarizados, >100 millones de células, todos con etiquetas de Cell Ontology; *Explorer* para navegar; herramienta *Gene Expression* para ver un marcador en todos los tejidos | Paquete `cellxgene_census` para descargar referencias anotadas como AnnData |
| [**Single Cell Portal**](https://singlecell.broadinstitute.org) (Broad Institute) | Cientos de estudios con gráficos interactivos y matrices/metadatos descargables | Descargar `.h5ad`/matrices → referencia para ingest/entrenar CellTypist |
| [**Cell Annotation Platform (CAP)**](https://celltype.info) | Sube tu `.h5ad`, registra **cada** tipo celular con evidencia de marcadores, sinónimos y términos de Cell Ontology; comparte/publica anotaciones | Exportar `adata.write_h5ad(...)` con etiquetas + genes marcadores |
| [**Human Cell Atlas Data Portal**](https://data.humancellatlas.org) | Datos crudos y procesados de proyectos del HCA | Descargar datos/referencias |
| [**Modelos de CellTypist**](https://www.celltypist.org/models) | Clasificadores preentrenados para muchos tejidos | `celltypist.models.download_models()` |
| [**Cell Ontology (CL)**](https://www.ebi.ac.uk/ols4/ontologies/cl) | IDs estándar para tipos celulares (`CL:0000236` = célula B) | Guarda los IDs en `adata.obs` para que otros reutilicen tus etiquetas |

**Buena práctica:** guarda un ID de Cell Ontology junto a cada etiqueta. CELLxGENE y CAP usan estos IDs, y hacen que
tus anotaciones sean comparables entre estudios.

In [ ]:
cell_ontology = {
    "CD4 T": ("CD4-positive, alpha-beta T cell", "CL:0000624"),
    "CD8 T": ("CD8-positive, alpha-beta T cell", "CL:0000625"),
    "NK": ("natural killer cell", "CL:0000623"),
    "B": ("B cell", "CL:0000236"),
    "CD14 Mono": ("CD14-positive monocyte", "CL:0001054"),
    "FCGR3A Mono": ("CD14-low, CD16-positive monocyte", "CL:0002396"),
    "DC": ("dendritic cell", "CL:0000451"),
    "Platelet": ("platelet", "CL:0000233"),
}
adata.obs["cell_type_ontology_term"] = adata.obs["cell_type"].map({k: v[0] for k, v in cell_ontology.items()})
adata.obs["cell_type_ontology_term_id"] = adata.obs["cell_type"].map({k: v[1] for k, v in cell_ontology.items()})
adata.obs[["cell_type", "cell_type_ontology_term", "cell_type_ontology_term_id"]].drop_duplicates().reset_index(drop=True)

🔎 **Extra: descargar una referencia de CZ CELLxGENE** (no se ejecuta en clase):
1. Navega <https://cellxgene.cziscience.com/datasets>, filtra por tejido/enfermedad, abre un conjunto de datos y copia su **dataset ID**.
2. Descárgalo como AnnData:
```python
!pip install -q cellxgene-census
import cellxgene_census
cellxgene_census.download_source_h5ad("<dataset_id>", to_path="reference.h5ad")
ref = sc.read_h5ad("reference.h5ad")      # etiquetas en ref.obs["cell_type"]
```
Luego úsala con `sc.tl.ingest`, o entrena tu propio modelo de CellTypist: `celltypist.train(ref, labels="cell_type")`.

## 8. ¿Coinciden los métodos? 🟢
La concordancia entre métodos independientes es tu evidencia. Las discrepancias señalan clústeres que requieren una
mirada más cercana (clústeres mixtos, dobletes, estados raros o activados).

In [ ]:
sc.pl.umap(adata, color=["cell_type", "panglaodb", "celltypist", "ingest"], ncols=2, wspace=0.7)

In [ ]:
import seaborn as sns
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, other in zip(axes, ["panglaodb", "celltypist", "ingest"]):
    tab = pd.crosstab(adata.obs["cell_type"], adata.obs[other], normalize="index")
    sns.heatmap(tab, cmap="Blues", ax=ax, cbar=False, vmin=0, vmax=1)
    ax.set(title=f"markers vs {other}", xlabel=other, ylabel="cell_type (markers)")
plt.tight_layout()

## 9. Guardar el objeto anotado 🟢

In [ ]:
adata.uns.pop("rank_genes_groups", None)   # se recalcula por tipo celular en el notebook 04
adata.write_h5ad(f"{WORKDIR}/results/pbmc3k_annotated.h5ad")
adata.obs["cell_type"].value_counts()

### 💡 Te toca a ti
Ejecuta CellTypist otra vez con el modelo más general `Immune_All_High.pkl` y compáralo con `cell_type`
usando `pd.crosstab`. ¿Qué etiquetas se vuelven más generales?

In [ ]:
# Tu código aquí

<details><summary><b>Ver solución</b></summary>

```python
models.download_models(model=["Immune_All_High.pkl"])
pred_high = celltypist.annotate(adata_ct, model="Immune_All_High.pkl", majority_voting=True,
                                over_clustering="leiden")
pd.crosstab(adata.obs["cell_type"], pred_high.predicted_labels["majority_voting"].values)
```
</details>

---
## ✅ ¡Listo!
Cada célula tiene nombre. Siguiente: **04 · Visualización, expresión diferencial y análisis posteriores**.